# 04b — LangGraph: Stateful Agents
**Domain:** IT / Incident Runbooks &nbsp;|&nbsp; **Time:** ~3 h  
**Key Concepts:** StateGraph, conditional branching, checkpointing, HITL

---
### Why LangGraph?
ReAct loops are great for simple tool use but hard to debug, persist, or interrupt.  
LangGraph models agent behaviour as a **directed graph of nodes + edges**, enabling:
- **Conditional branching** — go to node A or B based on state
- **Checkpointing** — resume mid-workflow after crash or human review
- **Human-in-the-loop** — pause and wait for approval before continuing


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated, List
    import operator
    print("✅ LangGraph available")
except ImportError:
    print("❌ pip install langgraph")


---
## 🔵 Example — Ex 04b-1: Minimal LangGraph Agent

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
from openai import OpenAI
import operator

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

class IncidentState(TypedDict):
    query:        str
    severity:     str          # "high" | "medium" | "low"
    step_count:   int
    final_answer: str

# ── Nodes ──────────────────────────────────────────────────────────────────
def classify_node(state: IncidentState) -> dict:
    resp = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "system", "content": "Reply with ONLY one word: high, medium, or low."},
            {"role": "user",   "content": f"Severity of: {state['query']}"},
        ],
        max_tokens=5,
    )
    sev = resp.choices[0].message.content.strip().lower()
    if sev not in ("high", "medium", "low"):
        sev = "medium"
    return {"severity": sev, "step_count": state.get("step_count", 0) + 1}

def escalate_node(state: IncidentState) -> dict:
    return {"final_answer": f"[ESCALATED] Paging on-call: {state['query']}"}

def resolve_node(state: IncidentState) -> dict:
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user",
                   "content": f"Give 3 resolution steps for: {state['query']}"}],
        max_tokens=150,
    )
    return {"final_answer": resp.choices[0].message.content}

# ── Routing ────────────────────────────────────────────────────────────────
def route(state: IncidentState) -> str:
    if state.get("step_count", 0) > 5:
        return END
    return "escalate" if state["severity"] == "high" else "resolve"

# ── Build ──────────────────────────────────────────────────────────────────
g = StateGraph(IncidentState)
g.add_node("classify", classify_node)
g.add_node("escalate", escalate_node)
g.add_node("resolve",  resolve_node)
g.set_entry_point("classify")
g.add_conditional_edges("classify", route, {"escalate": "escalate", "resolve": "resolve", END: END})
g.add_edge("escalate", END)
g.add_edge("resolve",  END)
app = g.compile()

result = app.invoke({"query": "Database unresponsive — all transactions failing",
                     "severity": "", "step_count": 0, "final_answer": ""})
print(f"Severity: {result['severity']}\nAnswer:   {result['final_answer'][:200]}")


---
## 🟡 Exercise — Ex 04b-2: Incident LangGraph with Branching

In [ ]:
show_exercise(
    "04b-2", "Incident LangGraph with 3-way branching",
    """Build a StateGraph with nodes:
  classify → high: escalate | medium: notify | low: auto_resolve
Each path ends at END. Add step_count guardrail (max 8).
Return dict with keys: severity, resolution, step_count.""",
    hints=[
        "TypedDict for state; Annotated[List[str], operator.add] for lists that accumulate",
        "add_conditional_edges('classify', routing_fn, {'high':'escalate', ...})",
        "Increment step_count in every node; check it in the routing function",
    ],
    checks=[
        "Graph compiles without error",
        "High severity → escalate path",
        "step_count <= 8 in all runs",
        "Result dict has 'severity' and 'resolution' keys",
    ],
)


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator, datetime

class IncidentV2(TypedDict):
    query:       str
    severity:    str
    diagnosis:   str
    resolution:  str
    step_count:  int

# TODO: define classify_v2, diagnose_v2, escalate_v2, notify_v2, auto_resolve_v2 nodes
# TODO: routing function
# TODO: compile incident_graph_v2

incident_graph_v2 = None   # ← replace with compiled graph

def run_incident(query: str) -> dict:
    # TODO: invoke incident_graph_v2 with initial state
    pass


In [ ]:
try:
    r_high = run_incident("CRITICAL: prod DB is completely down — all writes failing")
    r_low  = run_incident("Docs page has a typo on the homepage")
except Exception as e:
    r_high = r_low = None
    print(f"Error: {e}")

check([
    (incident_graph_v2 is not None,                      "Graph compiled"),
    (r_high is not None,                                 "High-severity run completed"),
    (r_low  is not None,                                 "Low-severity run completed"),
    (isinstance(r_high, dict) if r_high else False,      "Returns dict"),
    (r_high.get("step_count", 9) <= 8 if r_high else False, "step_count <= 8"),
    ("severity"   in (r_high or {}),                     "'severity' key present"),
    ("resolution" in (r_high or {}),                     "'resolution' key present"),
], exercise_id="04b-2")


In [ ]:
evaluate(run_incident,
         "LangGraph incident agent: classify severity, branch into 3 paths, step_count guardrail.",
         exercise_id="04b-2")


---
## 🔴 Challenge — Ex 04b-3: Human-in-the-Loop Interrupt

In [ ]:
show_exercise(
    "04b-3", "Human-in-the-loop approval before escalation",
    """Add an interrupt BEFORE the escalate node so a human can approve or reject.
If approved  → escalate.
If rejected  → fall back to auto_resolve.
Use graph.compile(interrupt_before=["escalate"]) and resume with app.invoke(None, config=config).
For demo: pass approve: bool as a parameter instead of real user input.""",
    hints=[
        "from langgraph.checkpoint.memory import MemorySaver",
        "Compile: app = graph.compile(checkpointer=MemorySaver(), interrupt_before=['escalate'])",
        "Resume:  app.invoke(None, config=config) after setting approval in state",
    ],
    checks=[
        "Graph compiled with interrupt_before=['escalate']",
        "approve=True  → escalate path taken",
        "approve=False → auto_resolve path taken",
    ],
)


In [ ]:
def run_with_hitl(query: str, approve: bool = True) -> dict:
    """Run incident graph with human-in-the-loop approval for high-severity incidents."""
    # TODO:
    # 1. compile graph with MemorySaver + interrupt_before=["escalate"]
    # 2. invoke until interrupt fires
    # 3. simulate human decision (approve param)
    # 4. update state and resume
    # 5. return final state
    pass


r_approved = run_with_hitl("CRITICAL: payment service completely down", approve=True)
r_rejected = run_with_hitl("CRITICAL: payment service completely down", approve=False)

print("Approved:", (r_approved or {}).get("resolution", "")[:100])
print("Rejected:", (r_rejected or {}).get("resolution", "")[:100])

check([
    (r_approved is not None, "HITL approved path completes"),
    (r_rejected is not None, "HITL rejected path completes"),
], exercise_id="04b-3")


In [ ]:
check([
    (run_incident("test") is not None,     "LangGraph agent works (04b-2)"),
    (run_with_hitl("test") is not None,    "HITL interrupt works (04b-3)"),
], exercise_id="04b-final")
progress()
